In [1]:
from multiprocessing import Queue
monitor_events_queue = Queue()

from dataclasses import dataclass

@dataclass
class Event:
    source: str       # отправитель
    destination: str  # получатель
    operation: str    # чего хочет (запрашиваемое действие)
    parameters: str   # с какими параметрами

In [2]:
from multiprocessing import Queue, Process
from multiprocessing.queues import Empty
import json

# формат управляющих команд для монитора
@dataclass
class ControlEvent:
    operation: str

# список разрешенных сочетаний сигналов светофора
# любые сочетания, отсутствующие в этом списке, запрещены
traffic_lights_allowed_configurations = [
    {"direction_1": "red", "direction_2": "green", "direction_1_arrow" : "off", "direction_2_arrow" : "on"},
    {"direction_1": "green", "direction_2": "red", "direction_1_arrow" : "on", "direction_2_arrow" : "off"},
    {"direction_1": "red", "direction_2": "red", "direction_1_arrow" : "off", "direction_2_arrow" : "off"},
    {"direction_1": "red", "direction_2": "yellow", "direction_1_arrow" : "off", "direction_2_arrow" : "on"},
    {"direction_1": "yellow", "direction_2": "yellow", "direction_1_arrow" : "on", "direction_2_arrow" : "on"},
    {"direction_1": "off", "direction_2": "off", "direction_1_arrow" : "off", "direction_2_arrow" : "off"},
    {"direction_1": "green", "direction_2": "red", "direction_1_arrow" : "on", "direction_2_arrow" : "off"},
    {"direction_1": "green", "direction_2": "yellow", "direction_1_arrow" : "on", "direction_2_arrow" : "on"},
    {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking", "direction_1_arrow" : "off", "direction_2_arrow" : "off"},
]


# Класс, реализующий поведение монитора безопасности
class Monitor(Process):

    def __init__(self, events_q: Queue):
        # вызываем конструктор базового класса
        super().__init__()
        self._events_q = events_q  # очередь событий для монитора (входящие сообщения)
        self._control_q = Queue()  # очередь управляющих команд (например, для остановки монитора)
        self._entity_queues = {}   # словарь очередей известных монитору сущностей
        self._force_quit = False   # флаг завершения работы монитора

    # регистрация очереди новой сущности
    def add_entity_queue(self, entity_id: str, queue: Queue):
        print(f"[монитор] регистрируем сущность {entity_id}")
        self._entity_queues[entity_id] = queue

    def _check_mode(self, mode_str: str) -> bool:
        mode_ok = False
        try:
            # извлечём структуру из строки, в случае ошибки запретим изменение режима
            print
            mode = json.loads(mode_str)
            # проверим входит ли запрашиваемый режим в список разрешённых
            print(f"[монитор] проверяем конфигурацию {mode}")


            if mode in traffic_lights_allowed_configurations:
                # такой режим найден, можно активировать
                mode_ok = True
        except:
            mode_ok = False
        return mode_ok

    # проверка политик безопасности
    def _check_policies(self, event):
        print(f'[монитор] обрабатываем событие {event}')

        # default deny: всё, что не разрешено, запрещено по умолчанию!
        authorized = False

        # проверка на входе, что это экземпляр класса Event,
        # т.е. имеет ожидаемый формат
        if not isinstance(event, Event):
            return False

        #
        #  политики безопасности
        #

        # пример политики безопасности
        if event.source == "ControlSystem" \
                and event.destination == "LightsGPIO" \
                and event.operation == "set_mode" \
                and self._check_mode(event.parameters):
            authorized = True

        if event.source == "ControlSystem" \
                and event.destination == "CitySystemConnector" \
                and event.operation == "local_log_save":
            authorized = True

        if event.source == "LightsGPIO" \
                and event.destination == "SelfDiagnosticsSystem" \
                and event.operation == "light_check" \
                and self._check_mode(event.parameters):
            authorized = True

        if event.source == "SelfDiagnosticsSystem" \
                and event.destination == "ControlSystem" \
                and event.operation == "save_traffic_light_log":
            authorized = True

        if event.source == "CitySystemConnector" \
                and event.destination == "ControlSystem" \
                and (event.operation == "set_regulated_mode" or event.operation == "set_unregulated_mode" or event.operation == "set_green_durations"):
            authorized = True

        if authorized is False:
            print("[монитор] событие не разрешено политиками безопасности")
        return authorized

    # выполнение разрешённого запроса
    # метод должен вызываться только после проверки политик безопасности
    def _proceed(self, event):
        print(f'[монитор] отправляем запрос {event}')
        try:
            # найдём очередь получателя события
            dst_q: Queue = self._entity_queues[event.destination]
            # и положим запрос в эту очередь
            dst_q.put(event)
        except  Exception as e:
            # например, запрос пришёл от или для неизвестной сущности
            print(f"[монитор] ошибка выполнения запроса {e}")

    # основной код работы монитора безопасности
    def run(self):
        print('[монитор] старт')

        # в цикле проверяет наличие новых событий,
        # выход из цикла по флагу _force_quit
        while self._force_quit is False:
            event = None
            try:
                # ожидание сделано неблокирующим,
                # чтобы можно было завершить работу монитора,
                # не дожидаясь нового сообщения
                event = self._events_q.get_nowait()

                # сюда попадаем только в случае получение события,
                # теперь нужно проверить политики безопасности
                authorized = self._check_policies(event)
                if authorized:
                    # если политиками запрос авторизован - выполняем
                    self._proceed(event)
            except Empty:
                # сюда попадаем, если новых сообщений ещё нет,
                # в таком случае немного подождём
                sleep(0.5)
            except Exception as e:
                # что-то пошло не так, выведем сообщение об ошибке
                print(f"[монитор] ошибка обработки {e}, {event}")
            self._check_control_q()
        print('[монитор] завершение работы')

    # запрос на остановку работы монитора безопасности для завершения работы
    # может вызываться вне процесса монитора
    def stop(self):
        # поскольку монитор работает в отдельном процессе,
        # запрос помещается в очередь, которая проверяется из процесса монитора
        request = ControlEvent(operation='stop')
        self._control_q.put(request)

    # проверка наличия новых управляющих команд
    def _check_control_q(self):
        try:
            request: ControlEvent = self._control_q.get_nowait()
            print(f"[монитор] проверяем запрос {request}")
            if isinstance(request, ControlEvent) and request.operation == 'stop':
                # поступил запрос на остановку монитора, поднимаем "красный флаг"
                self._force_quit = True
        except Empty:
            # никаких команд не поступило, ну и ладно
            pass

In [3]:
from multiprocessing import Queue, Process
import json


class ControlSystem(Process):

    def __init__(self, monitor_queue: Queue):
        # вызываем конструктор базового класса
        super().__init__()
        # мы знаем только очередь монитора безопасности для взаимодействия с другими сущностями
        # прямая отправка сообщений в другую сущность запрещена в концепции FLASK
        self.monitor_queue = monitor_queue
        # создаём собственную очередь, в которую монитор сможет положить сообщения для этой сущности
        self._own_queue = Queue()
        self.control_q = Queue()
        self.current_mode_type = None
        self.current_mode = None
        self.green_duration = 10
        self.force_quit = False


    # выдаёт собственную очередь для взаимодействия
    def entity_queue(self):
        return self._own_queue

    # основной код сущности
    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        print(f'[{self.__class__.__name__}] отправляем тестовый запрос')
        traffic_modes = [
            {"direction_1": "red", "direction_2": "green", "direction_1_arrow" : "off", "direction_2_arrow" : "on"},
            {"direction_1": "green", "direction_2": "red", "direction_1_arrow" : "on", "direction_2_arrow" : "off"},
        ]
        while self.force_quit is False:
            try:
                event = self._own_queue.get_nowait()
                authorized = True # self.check_policies(event)
                if event.operation == "set_regulated_mode":
                    self.current_mode_type = "regulated"

                if event.operation == "set_unregulated_mode":
                    self.current_mode_type = "unregulated"
                    self.current_mode = None
                    unregulated_mod = {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking", "direction_1_arrow" : "off", "direction_2_arrow" : "off"}
                    unregulated_mod_set_event = Event(source=self.__class__.__name__,
                          destination='LightsGPIO',
                          operation='set_mode',
                          parameters=json.dumps(unregulated_mod)
                          )
                    self.monitor_queue.put(unregulated_mod_set_event)
                if event.operation == "set_green_durations":
                    self.green_duration = event.parameters["duration"]
                    print(f"[{self.__class__.__name__}] установлено новая длительность зеленого {self.green_duration}")
                    

                if event.operation == "save_traffic_light_log":
                    parameters = json.loads(event.parameters)
                    print(f"[{self.__class__.__name__}] Сохранен лог {parameters['mod']} с результатом {parameters['status']}")
                    new_event = Event(source=self.__class__.__name__,
                          destination='CitySystemConnector',
                          operation='local_log_save',
                          parameters=event.parameters
                          )
                    self.monitor_queue.put(new_event)
                    
            except Empty:
                sleep(0.5)

            if self.current_mode_type == "regulated":
                regulated_mod = traffic_modes[0] if self.current_mode == traffic_modes[1] or self.current_mode == None else traffic_modes[1]
                regulated_mod_set_light_event = Event(source=self.__class__.__name__,
                      destination='LightsGPIO',
                      operation='set_mode',
                      parameters=json.dumps(regulated_mod)
                      )
                self.monitor_queue.put(regulated_mod_set_light_event)
                self.current_mode = regulated_mod
                sleep(self.green_duration)
            self.check_control_q()
        print(f'[{self.__class__.__name__}] завершение работы')


    def stop(self):
        request = ControlEvent(operation='stop_controle_system')
        self.control_q.put(request)

    def check_control_q(self):
        try:
            request: ControlEvent = self.control_q.get_nowait()
            print(f"[{self.__class__.__name__}] проверяем запрос {request}")
            if isinstance(request, ControlEvent) and request.operation == 'stop_controle_system':
                self.force_quit = True
        except Empty:
            pass

In [4]:
from multiprocessing import Queue, Process
from time import sleep


class LightsGPIO(Process):

    def __init__(self, monitor_queue: Queue):
        # вызываем конструктор базового класса
        super().__init__()
        # мы знаем только очередь монитора безопасности для взаимодействия с другими сущностями
        # прямая отправка сообщений в другую сущность запрещена в концепции FLASK
        self.monitor_queue = monitor_queue
        # создаём собственную очередь, в которую монитор сможет положить сообщения для этой сущности
        self._own_queue = Queue()
        self.control_q = Queue()
        self.force_quit = False

    def entity_queue(self):
        return self._own_queue

    # основной код сущности
    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        while self.force_quit is False:
            try:
                event: Event = self._own_queue.get_nowait()
                if event.operation == "set_mode":
                    print(f"[{self.__class__.__name__}] {event.source} запрашивает изменение режима {event.parameters}")
                    print(f"[{self.__class__.__name__}] новый режим: {event.parameters}!")

                event_log = Event(source=self.__class__.__name__,
                      destination='SelfDiagnosticsSystem',
                      operation='light_check',
                      parameters = event.parameters
                      )
                self.monitor_queue.put(event_log)

            except Empty:
                sleep(0.5)
            self.check_queue_for_exit()

        print(f'[{self.__class__.__name__}] завершение работы')

    def stop(self):
        request = ControlEvent(operation='light_gpio_stop')
        self.control_q.put(request)

    def check_queue_for_exit(self):
        try:
            request: ControlEvent = self.control_q.get_nowait()
            print(f"[{self.__class__.__name__}] проверяем запрос на выключение {request}")
            if isinstance(request, ControlEvent) and request.operation == 'light_gpio_stop':
                self.force_quit = True
        except Empty:
            pass

In [5]:
from multiprocessing import Queue, Process
import json


class SelfDiagnosticsSystem(Process):

    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.control_q = Queue()
        self.force_quit = False

    def entity_queue(self):
        return self._own_queue


    def run(self):
          print(f'[{self.__class__.__name__}] старт')
          while self.force_quit is False:
              try:
                  event: Event = self._own_queue.get_nowait()
                  if event.operation == "light_check":
                      print(f"[{self.__class__.__name__}] {event.source} запрашивает проверку цвета светофора {event.parameters}")
                      all_ok = False
                      if True: # Условия диагностики:
                          all_ok = True

                      if all_ok:
                          status = 1
                      else:
                          status = 0
                          
                      parameters = {"status" : status, "mod" : json.loads(event.parameters)}    
                      event = Event(source=self.__class__.__name__,
                          destination='ControlSystem',
                          operation='save_traffic_light_log',
                          parameters=json.dumps(parameters)
                          )
                      self.monitor_queue.put(event)

              except Empty:
                  sleep(0.5)
              self.check_queue_for_exit()
          print(f'[{self.__class__.__name__}] завершение работы')

    def stop(self):
        request = ControlEvent(operation='self_diagnostics_stop')
        self.control_q.put(request)

    def check_queue_for_exit(self):
        try:
            request: ControlEvent = self.control_q.get_nowait()
            print(f"[{self.__class__.__name__}] проверяем запрос на выключение {request}")
            if isinstance(request, ControlEvent) and request.operation == 'self_diagnostics_stop':
                self.force_quit = True
        except Empty:
            pass


In [6]:
from multiprocessing import Queue, Process
import json
import random

class CitySystemConnector(Process):

    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.control_q = Queue()
        self.force_quit = False

    def entity_queue(self):
        return self._own_queue


    def run(self):
          print(f'[{self.__class__.__name__}] старт')
          while self.force_quit is False:
              try:
                  event: Event = self._own_queue.get_nowait()
                  if event.operation == "local_log_save":
                      print(f"[{self.__class__.__name__}] {event.source} запрашивает сохранение лога в [{self.__class__.__name__}], лог: {event.parameters}")
                      print(f"[{self.__class__.__name__}] Сохранен лог {event.parameters}")
                  
              except Empty:
                  sleep(0.5)
              
              random_event = random.randint(0, 2)
              if random_event == 0:
                  event = Event(source=self.__class__.__name__,
                          destination='ControlSystem',
                          operation='set_regulated_mode',
                          parameters = None
                          )
              elif random_event == 1:
                  event = Event(source=self.__class__.__name__,
                          destination='ControlSystem',
                          operation='set_unregulated_mode',
                          parameters = None
                          )
              elif random_event == 2:
                  event = Event(source=self.__class__.__name__,
                          destination='ControlSystem',
                          operation='set_green_durations',
                          parameters = {"duration" : random.randint(10, 20)}
                          )
              self.monitor_queue.put(event)
              self.check_queue_for_exit()
              sleep(25)
          print(f'[{self.__class__.__name__}] завершение работы')

    def stop(self):
        request = ControlEvent(operation='city_systemconnector_stop')
        self.control_q.put(request)

    def check_queue_for_exit(self):
        try:
            request: ControlEvent = self.control_q.get_nowait()
            print(f"[{self.__class__.__name__}] проверяем запрос на выключение {request}")
            if isinstance(request, ControlEvent) and request.operation == 'city_systemconnector_stop':
                self.force_quit = True
        except Empty:
            pass


In [7]:
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue)
сity_system_connector = CitySystemConnector(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue)
self_diagnostics_system = SelfDiagnosticsSystem(monitor_events_queue)

In [8]:
monitor.add_entity_queue(сity_system_connector.__class__.__name__, сity_system_connector.entity_queue())
monitor.add_entity_queue(control_system.__class__.__name__, control_system.entity_queue())
monitor.add_entity_queue(lights_gpio.__class__.__name__, lights_gpio.entity_queue())
monitor.add_entity_queue(self_diagnostics_system.__class__.__name__, self_diagnostics_system.entity_queue())

[монитор] регистрируем сущность CitySystemConnector
[монитор] регистрируем сущность ControlSystem
[монитор] регистрируем сущность LightsGPIO
[монитор] регистрируем сущность SelfDiagnosticsSystem


In [9]:
monitor.start()
сity_system_connector.start()
control_system.start()
lights_gpio.start()
self_diagnostics_system.start()
sleep(2)

[монитор] старт
[CitySystemConnector] старт
[ControlSystem] старт
[ControlSystem] отправляем тестовый запрос
[LightsGPIO] старт
[SelfDiagnosticsSystem] старт
[монитор] обрабатываем событие Event(source='CitySystemConnector', destination='ControlSystem', operation='set_green_durations', parameters={'duration': 15})
[монитор] отправляем запрос Event(source='CitySystemConnector', destination='ControlSystem', operation='set_green_durations', parameters={'duration': 15})
[ControlSystem] установлено новая длительность зеленого 15
[монитор] обрабатываем событие Event(source='CitySystemConnector', destination='ControlSystem', operation='set_regulated_mode', parameters=None)
[монитор] отправляем запрос Event(source='CitySystemConnector', destination='ControlSystem', operation='set_regulated_mode', parameters=None)
[монитор] обрабатываем событие Event(source='ControlSystem', destination='LightsGPIO', operation='set_mode', parameters='{"direction_1": "red", "direction_2": "green", "direction_1_ar

In [10]:
monitor.stop()
self_diagnostics_system.stop()
control_system.stop()
сity_system_connector.stop()
lights_gpio.stop()
monitor.stop()